In [1]:
#1
import os
os.chdir("/kaggle/working")
if not os.path.isdir("CatVTON"):
    !git clone https://github.com/Zheng-Chong/CatVTON.git
%cd /kaggle/working/CatVTON
!grep -vi '^gradio' requirements.txt | grep -vi '^huggingface_hub' > requirements_notebook.txt
!pip install -q -r requirements_notebook.txt
!pip install -q "huggingface_hub>=0.34.0,<2.0" fvcore iopath yacs pycocotools omegaconf cloudpickle av
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio

/kaggle/working/CatVTON
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Cannot install -r requirements_notebook.txt (line 14), -r requirements_notebook.txt (line 3) and -r requirements_notebook.txt (line 4) because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [2]:
#2
import os, time, io, glob
import torch
from PIL import Image
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

def find_local_dir(root, folder_name):
    """Tìm folder_name bên trong root, trả về path nếu tồn tại và có file."""
    if not os.path.isdir(root):
        return None
    for m in glob.glob(os.path.join(root, "**", folder_name), recursive=True):
        if os.path.isdir(m) and os.listdir(m):
            return m
    return None

DATASET_ROOT = "/kaggle/input/catvton-weights"
WORKING_ROOT = "/kaggle/working/catvton_weights"

catvton_local = find_local_dir(DATASET_ROOT, "CatVTON") or os.path.join(WORKING_ROOT, "CatVTON")
if os.path.isdir(catvton_local) and os.listdir(catvton_local):
    repo_path = catvton_local
    print(f"[cache] Dùng CatVTON weights có sẵn: {repo_path}")
else:
    print("[download] Chưa có cache, tải CatVTON từ HF Hub...")
    repo_path = snapshot_download(repo_id="zhengchong/CatVTON", local_dir=catvton_local)

sd_local = find_local_dir(DATASET_ROOT, "sd-inpainting") or os.path.join(WORKING_ROOT, "sd-inpainting")
if os.path.isdir(sd_local) and os.listdir(sd_local):
    base_ckpt_path = sd_local
    print(f"[cache] Dùng SD-inpainting weights có sẵn: {base_ckpt_path}")
else:
    print("[download] Chưa có cache, tải SD-inpainting từ HF Hub...")
    base_ckpt_path = snapshot_download(repo_id="booksforcharlie/stable-diffusion-inpainting", local_dir=sd_local)

pipeline = CatVTONPipeline(
    base_ckpt=base_ckpt_path,
    attn_ckpt=repo_path,
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device="cuda",
    skip_safety_check=True,
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
automasker = AutoMasker(densepose_ckpt=os.path.join(repo_path, "DensePose"), schp_ckpt=os.path.join(repo_path, "SCHP"), device="cuda")

# --- [OPT-1] Doi sang DPM++ 2M Karras scheduler ---
from diffusers import DPMSolverMultistepScheduler
pipeline.noise_scheduler = DPMSolverMultistepScheduler.from_config(
    pipeline.noise_scheduler.config,
    use_karras_sigmas=True,
    algorithm_type="dpmsolver++"
)
print("[OPT] Scheduler: DPM++ 2M Karras - OK")

# --- [OPT-2] Bat xFormers memory efficient attention ---
try:
    pipeline.unet.enable_xformers_memory_efficient_attention()
    print("[OPT] xFormers memory efficient attention - OK")
except Exception as e:
    print(f"[WARN] xFormers not available: {e}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


[cache] Dùng CatVTON weights có sẵn: /kaggle/working/catvton_weights/CatVTON
[cache] Dùng SD-inpainting weights có sẵn: /kaggle/working/catvton_weights/sd-inpainting


An error occurred while trying to fetch /kaggle/working/catvton_weights/sd-inpainting: Error no file named diffusion_pytorch_model.safetensors found in directory /kaggle/working/catvton_weights/sd-inpainting.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


[OPT] Scheduler: DPM++ 2M Karras - OK
[WARN] xFormers not available: Refer to https://github.com/facebookresearch/xformers for more information on how to install xformers


In [ ]:
#3
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

@app.post("/api/tryon")
async def tryon_api(person_image: UploadFile = File(...), cloth_image: UploadFile = File(...), category: str = Form("overall")):
    person_bytes = await person_image.read()
    cloth_bytes = await cloth_image.read()
    person_img = Image.open(io.BytesIO(person_bytes)).convert("RGB")
    cloth_img = Image.open(io.BytesIO(cloth_bytes)).convert("RGB")
    
    WIDTH, HEIGHT = 768, 1024
    person_img = resize_and_crop(person_img, (WIDTH, HEIGHT))
    cloth_img = resize_and_padding(cloth_img, (WIDTH, HEIGHT))
    
    mask = automasker(person_img, category)["mask"]
    mask = mask_processor.blur(mask, blur_factor=9)
    
    result = pipeline(image=person_img, condition_image=cloth_img, mask=mask, num_inference_steps=20, guidance_scale=4.5)[0]
    out_path = os.path.join(OUTPUT_DIR, f"result_{int(time.time())}.png")
    result.save(out_path)
    return FileResponse(out_path, media_type="image/png")

# ĐIỀN STATIC DOMAIN VÀ TOKEN NGROK CỦA BẠN VÀO ĐÂY
ngrok.set_auth_token("3B91485xasWPb3CYuMwP0qb76S7_7vgxKozCzYTYdnDCtaCFY")
public_url = ngrok.connect(8000, domain="cactaceous-tatum-semiadhesively.ngrok-free.dev").public_url
print(f"API SẴN SÀNG TẠI: {public_url}/api/tryon")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

API SẴN SÀNG TẠI: https://cactaceous-tatum-semiadhesively.ngrok-free.dev/api/tryon


INFO:     Started server process [338]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
100%|██████████| 20/20 [00:40<00:00,  2.03s/it]


INFO:     183.80.56.6:0 - "POST /api/tryon HTTP/1.1" 200 OK


 90%|█████████ | 18/20 [00:37<00:04,  2.26s/it]